# Pilhas (Stacks)

## Objetivos da aula

Ao final deste exemplo, voce devera compreender:

- o principio **LIFO** (*Last In, First Out*);
- o papel do topo em uma pilha;
- as operacoes `push`, `pop`, `peek` e `isEmpty`;
- como implementar uma pilha dinamica com lista encadeada;
- como cuidar da memoria alocada com `malloc` e `free`.

Uma **pilha** e uma estrutura de dados linear em que a insercao e a remocao acontecem pela mesma extremidade, chamada **topo**. O ultimo elemento inserido e o primeiro a ser removido.

A analogia mais comum e uma pilha de pratos: um prato novo e colocado no topo e somente o prato que esta no topo pode ser retirado sem remover os que estao acima dele.

## Regra LIFO e funcionamento

A regra de uma pilha e chamada **LIFO**: *Last In, First Out*, ou seja, o ultimo elemento que entra e o primeiro que sai.

```text
push(10) -> [10]
push(20) -> [10, 20]
push(30) -> [10, 20, 30]
pop()    -> remove 30
```

Depois do `pop`, a pilha fica `[10, 20]`. O valor `30` estava no topo porque foi o ultimo a ser inserido.

Uma pilha nao permite remover diretamente um elemento do meio. Para chegar a ele, primeiro seria necessario retirar todos os elementos que estao acima dele.

## Operacoes fundamentais

| Operacao | Finalidade | Efeito na pilha |
|---|---|---|
| `push` | Inserir um valor | Coloca o valor no topo. |
| `pop` | Remover um valor | Retira o valor que esta no topo. |
| `peek` ou `top` | Consultar | Le o topo sem remove-lo. |
| `isEmpty` | Verificar o estado | Informa se a pilha esta vazia. |

Quando tentamos retirar um elemento de uma pilha vazia, ocorre **underflow**. Em uma pilha implementada com vetor de tamanho fixo, tentar inserir quando nao ha espaco causa **overflow**.

As operacoes principais atuam somente no topo. Por isso, `push`, `pop` e `peek` geralmente possuem complexidade `O(1)`.

## Implementacao escolhida: pilha dinamica

Neste exemplo, a pilha sera implementada com uma **lista encadeada**. Cada elemento sera armazenado em um no, formado por:

```text
topo -> [dado | prox] -> [dado | prox] -> NULL
```

- `dado` guarda o valor do elemento.
- `prox` aponta para o no que estava anteriormente no topo.
- `topo` aponta para o no mais recente.
- `NULL` indica que nao existe outro no abaixo dele.

A pilha dinamica nao possui uma capacidade fixa. Cada `push` aloca um novo no e cada `pop` libera o no removido. A capacidade fica limitada pela memoria disponivel.

A implementacao completa aparece na proxima celula. Depois dela, cada parte do codigo sera explicada individualmente.

In [32]:
#include <stdbool.h>
#include <stdio.h>
#include <stdlib.h>

typedef struct NO {
    int dado;
    struct NO *prox;
} Elemento;

typedef struct Pilhas {
    Elemento *topo;
} Pilhas;

void inicializar(Pilhas *p) {
    p->topo = NULL;
}

bool estaVazia(const Pilhas *p) {
    return p->topo == NULL;
}

void inserir(Pilhas *p, int valor) {
    Elemento *novo = malloc(sizeof(Elemento));
    if (novo == NULL) {
        printf("Erro ao alocar memoria!\n");
        return;
    }

    novo->dado = valor;
    novo->prox = p->topo;
    p->topo = novo;
    printf("Elemento %d inserido com sucesso!\n", valor);
}

bool remover(Pilhas *p, int *valor) {
    if (estaVazia(p)) {
        printf("Pilha vazia!\n");
        return false;
    }

    Elemento *noRemovido = p->topo;
    *valor = noRemovido->dado;
    p->topo = noRemovido->prox;
    free(noRemovido);
    return true;
}

bool consultarTopo(const Pilhas *p, int *valor) {
    if (estaVazia(p)) {
        return false;
    }

    *valor = p->topo->dado;
    return true;
}

void exibir(const Pilhas *p) {
    if (estaVazia(p)) {
        printf("Pilha vazia!\n");
        return;
    }else{
        Elemento *aux = p->topo;
        printf("Elementos da pilha, do topo para a base:\n");
        do {
            printf("%d\n", aux->dado);
            aux = aux->prox;
        } while (aux != NULL);
    }
}
void liberar(Pilhas *p) {
    int valor;
    while (remover(p, &valor)) {
    }
}

int main(void) {
    Pilhas pilha;
    int valor;

    inicializar(&pilha);
    exibir(&pilha);

    inserir(&pilha, 5);
    inserir(&pilha, 8);
    inserir(&pilha, 3);
    exibir(&pilha);

    if (consultarTopo(&pilha, &valor)) {
        printf("Topo: %d\n", valor);
    }

    if (remover(&pilha, &valor)) {
        printf("Elemento removido: %d\n", valor);
    }
    exibir(&pilha);

    liberar(&pilha);
    return 0;
}

Pilha vazia!
Elemento 5 inserido com sucesso!
Elemento 8 inserido com sucesso!
Elemento 3 inserido com sucesso!
Elementos da pilha, do topo para a base:
3
8
5
Topo: 3
Elemento removido: 3
Elementos da pilha, do topo para a base:
8
5
Pilha vazia!


## Explicacao do codigo, funcao por funcao

### 1. Bibliotecas

```c
#include <stdbool.h>
#include <stdio.h>
#include <stdlib.h>
```

- `stdio.h` fornece `printf`.
- `stdbool.h` fornece o tipo `bool`, alem de `true` e `false`.
- `stdlib.h` fornece `malloc` e `free`, usados para gerenciar memoria dinamica.

### 2. Estrutura do no

```c
typedef struct NO {
    int dado;
    struct NO *prox;
} Elemento;
```

Cada `Elemento` e um no da lista encadeada. O campo `dado` guarda o valor e `prox` aponta para o no que estava anteriormente no topo. O uso de `struct NO *` e necessario porque o tipo ainda esta sendo declarado.

### 3. Estrutura da pilha

```c
typedef struct Pilhas {
    Elemento *topo;
} Pilhas;
```

A pilha armazena apenas um ponteiro para o topo. Quando `topo == NULL`, nao existe nenhum no e a pilha esta vazia.

### 4. Inicializacao

```c
void inicializar(Pilhas *p) {
    p->topo = NULL;
}
```

A funcao recebe o endereco da pilha e define o topo como `NULL`. O operador `->` acessa um campo por meio de um ponteiro para uma estrutura.

### 5. Verificacao de pilha vazia

```c
bool estaVazia(const Pilhas *p) {
    return p->topo == NULL;
}
```

A funcao retorna `true` quando o topo nao aponta para nenhum no. O modificador `const` informa que a funcao apenas consulta a pilha.

### 6. Insercao: `inserir`

```c
void inserir(Pilhas *p, int valor) {
    Elemento *novo = malloc(sizeof(Elemento));
    if (novo == NULL) {
        printf("Erro ao alocar memoria!\n");
        return;
    }

    novo->dado = valor;
    novo->prox = p->topo;
    p->topo = novo;
}
```

A funcao realiza quatro passos:

1. Reserva memoria para um novo no.
2. Verifica se a alocacao foi realizada.
3. Guarda o valor e conecta o novo no ao topo anterior.
4. Atualiza o topo para apontar para o novo no.

Como a insercao ocorre no inicio da lista, nenhum outro no precisa ser percorrido. O custo e `O(1)`.

### 7. Remocao: `remover`

```c
bool remover(Pilhas *p, int *valor) {
    if (estaVazia(p)) {
        return false;
    }

    Elemento *noRemovido = p->topo;
    *valor = noRemovido->dado;
    p->topo = noRemovido->prox;
    free(noRemovido);
    return true;
}
```

A funcao verifica se existe um elemento, guarda o no do topo, copia seu dado para `valor`, move o topo para o proximo no e libera a memoria antiga. O ponteiro `valor` permite devolver o elemento removido ao chamador.

### 8. Consulta do topo

```c
bool consultarTopo(const Pilhas *p, int *valor) {
    if (estaVazia(p)) {
        return false;
    }

    *valor = p->topo->dado;
    return true;
}
```

A consulta le o valor do topo sem alterar a estrutura. Ela retorna `false` se a pilha estiver vazia e `true` quando a consulta for realizada.

### 9. Exibicao

```c
void exibir(const Pilhas *p) {
    for (Elemento *aux = p->topo; aux != NULL; aux = aux->prox) {
        printf("%d\n", aux->dado);
    }
}
```

O ponteiro auxiliar percorre os nós do topo até `NULL`. A funcao mostra os valores na ordem em que seriam removidos, isto e, do topo para a base. Seu custo e `O(n)`.

### 10. Liberacao da memoria

```c
void liberar(Pilhas *p) {
    int valor;
    while (remover(p, &valor)) {
    }
}
```

A funcao remove todos os nos restantes. Isso evita vazamento de memoria quando o programa termina com elementos na pilha.

### 11. Funcao `main`

A funcao `main` inicializa duas variaveis, insere `5`, `8` e `3` na pilha, consulta o topo, remove um elemento e exibe os estados. O valor `3` e o topo porque foi inserido por ultimo.

## Simulacao da execucao

Depois das insercoes:

```text
topo -> [3] -> [8] -> [5] -> NULL
```

A consulta retorna `3`. Depois do `pop`:

```text
topo -> [8] -> [5] -> NULL
```

A ordem de remocao sera `3`, `8`, `5`, confirmando o principio LIFO.

## Complexidade e memoria

| Operacao | Complexidade | Memoria |
|---|---:|---:|
| `inserir` | `O(1)` | Um novo no |
| `remover` | `O(1)` | Libera um no |
| `consultarTopo` | `O(1)` | Nenhuma alocacao |
| `exibir` | `O(n)` | Um ponteiro auxiliar |
| `liberar` | `O(n)` | Libera todos os nos |

A pilha encadeada cresce conforme a necessidade, mas cada no usa memoria adicional para armazenar o ponteiro `prox`.

## Exercicios



### Histórico de Navegação do Navegador
* Você foi contratado por uma startup que está desenvolvendo um navegador web do zero. A primeira funcionalidade a ser entregue é a navegação por histórico: o usuário precisa conseguir voltar para páginas" visitadas anteriormente e avançar novamente, exatamente como nos navegadores que usamos no dia a dia.
* O time de produto definiu o comportamento esperado:
* Quando o usuário visita uma página nova, ela é adicionada ao histórico e passa a ser a página atual.
* O botão Voltar leva o usuário à página anterior. A página atual não é perdida - ela fica guardada para permitir o Avançar.
* O botão Avançar refaz uma navegação que foi desfeita pelo Voltar.
* Regra crítica: quando o usuário visita uma página nova, todo o histórico de "Avançar" é descartado. Isso acontece porque, ao navegar para um lugar novo, os navegadores reais invalidam as páginas que estavam "à frente"

#### O problema
* Implemente, em linguagem C, um sistema de navegação com histórico usando pilha dinâmica (implementada com lista encadeada e alocação de memória via malloc/free). O sistema deve suportar as seguintes operações
* Operação visitar (url)
**comportamento Define url como página atual e a adiciona ao histórico. Esvazia o histórico de avançar.**
* operacao voltar()
**comportamento Retorna à página anterior. A página atual vai para o histórico de avançar. Se não houver página anterior, nada acontece**(exibe aviso).
* operacao avancar()
**comportamento Refaz a navegação desfeita. A página volta a ser a atual. Se não houver página para avançar, nada acontece (exibe aviso).**
* exibirAtual()
**Mostra a página atual na tela.**